### Create Age column based on DOB column (Age when crime was committed)

In [1]:
import pandas as pd 
import os

In [2]:
notebook_dir = os.getcwd()
data_dir = os.path.join(notebook_dir, '..', 'data')
checkpoint_dir = os.path.join(data_dir, 'checkpoints')

In [3]:
cp8 = pd.read_csv("/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/streamlit-app/checkpoint8_combined_data.csv")

In [4]:
cp8

,Incident #,Date,Type,Location,Arrested,DOB,Charges,Disposition,Location Prefix,raw_address,cleaned_address,latitude,longitude,geocode_confidence,geocoded_at,coord_valid,person_id,category,Year,crime_severity
0,26025341,2026-06-01 23:53:00,NOISE ORDINANCE,ALLSTON ST & HOWARD ST,No,NaN,NaN,SERVICED - SEE NOTES,NaN,ALLSTON ST & HOWARD ST,"ALLSTON ST & HOWARD ST, Lawrence, MA",42.711489,-71.146258,8.0,2026-06-28T02:47:17.094562+00:00,True,NaN,Other,2026,Non-Serious
1,26025340,2026-06-01 23:52:00,SEX OFFENSE,75 MANCHESTER ST,No,NaN,NaN,REPORT REQUIRED,NaN,75 MANCHESTER ST,"75 MANCHESTER ST, Lawrence, MA",42.712262,-71.179138,10.0,2026-06-27T19:47:43.969354+00:00,True,NaN,Other,2026,Non-Serious
2,26025338,2026-06-01 23:29:00,EXTRA CHECK,"SOUTH LAWRENCE EAST SCHOOL, 165 CRAWFORD ST",No,NaN,NaN,SERVICED - SEE NOTES,NaN,"SOUTH LAWRENCE EAST SCHOOL, 165 CRAWFORD ST","SOUTH LAWRENCE EAST SCHOOL, Lawrence, MA",42.707040,-71.163110,7.0,2026-06-30T15:19:59.495398+00:00,True,NaN,Other,2026,Non-Serious
3,26025337,2026-06-01 23:16:00,911 HANG-UP,98 OSGOOD ST,No,NaN,NaN,SERVICED - SEE NOTES,NaN,98 OSGOOD ST,"98 OSGOOD ST, Lawrence, MA",42.696509,-71.148493,10.0,2026-06-28T02:37:44.357795+00:00,True,NaN,Other,2026,Non-Serious
4,26025336,2026-06-01 23:15:00,MV STOP,20 ISLAND ST,No,NaN,NaN,SERVICED - SEE NOTES,NaN,20 ISLAND ST,"20 ISLAND ST, Lawrence, MA",42.706226,-71.149554,10.0,2026-06-20T19:58:08.588338+00:00,True,NaN,Other,2026,Non-Serious
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26497,26035542,2026-07-31 15:25:00,CONTINUED INVESTIGATION,280 MERRIMACK ST,No,NaN,NaN,SERVICED - SEE NOTES,NaN,280 MERRIMACK ST,"280 MERRIMACK ST, Lawrence, MA",42.703054,-71.147583,10.0,2026-06-23T00:36:41.207306+00:00,True,NaN,Other,2026,Non-Serious
26498,26035541,2026-07-31 15:23:00,CONTINUED INVESTIGATION,176 ANDOVER ST,No,NaN,NaN,SERVICED - SEE NOTES,NaN,176 ANDOVER ST,"176 ANDOVER ST, Lawrence, MA",42.696173,-71.152559,10.0,2026-06-20T06:08:40.151087+00:00,True,NaN,Other,2026,Non-Serious
26499,26035540,2026-07-31 15:22:00,HIT & RUN MOTOR VEHICLE,23 POPLAR ST,No,NaN,NaN,SERVICED - SEE NOTES,NaN,23 POPLAR ST,"23 POPLAR ST, Lawrence, MA",42.718330,-71.169128,10.0,2026-06-22T21:50:35.404177+00:00,True,NaN,Other,2026,Non-Serious
26500,26035539,2026-07-31 15:17:00,UNATTENDED CHILD,82 KINGSTON ST,No,NaN,NaN,REPORT REQUIRED,NaN,82 KINGSTON ST,"82 KINGSTON ST, Lawrence, MA",42.695033,-71.167862,10.0,2026-06-28T00:26:24.766155+00:00,True,NaN,Other,2026,Non-Serious


In [5]:
# work on a copy
df = cp8.copy()

# Optional, tidy column names and Arrested values
df.columns = df.columns.str.strip()
if 'Arrested' in df.columns:
    df['Arrested'] = df['Arrested'].astype(str).str.strip().str.title()

# Keep originals for debugging
df['_DOB_raw'] = df['DOB'].astype(str)
df['_Date_raw'] = df['Date'].astype(str)

# Parse to datetime
df['DOB_dt'] = pd.to_datetime(df['DOB'], errors='coerce', utc=False)
df['Date_dt'] = pd.to_datetime(df['Date'], errors='coerce', utc=False)

# Summary of parsing results
dob_total = df['DOB'].shape[0]
dob_nonnull = df['DOB'].notna().sum()
dob_failed = df['DOB_dt'].isna() & df['DOB'].notna()
date_failed = df['Date_dt'].isna() & df['Date'].notna()

print("DOB non-null count:", dob_nonnull)
print("DOB failed to parse count:", dob_failed.sum())
print("Date failed to parse count:", date_failed.sum())

# Show a few examples of bad DOB and Date entries
if dob_failed.any():
    print("\nSample problematic DOB values:")
    print(df.loc[dob_failed, '_DOB_raw'].drop_duplicates().head(20).to_list())

if date_failed.any():
    print("\nSample problematic Date values:")
    print(df.loc[date_failed, '_Date_raw'].drop_duplicates().head(20).to_list())

# If you want to enforce the standardized columns back into cp8
# only do this once you are happy with the parse results
cp8['DOB'] = df['DOB_dt']
cp8['Date'] = df['Date_dt']

# Final sanity check
print("\nDtypes after standardization:")
print(cp8[['DOB', 'Date']].dtypes)


DOB non-null count: 0
DOB failed to parse count: 0
Date failed to parse count: 0

Dtypes after standardization:
DOB     datetime64[ns]
Date    datetime64[ns]
dtype: object


In [6]:
import numpy as np

In [7]:
# mask for rows we (those who were charged, assume charged if DOB is present)
mask_charged = cp8['DOB'].notna()

# compute age at offense date with birthday adjustment
before_birthday = (
    (cp8.loc[mask_charged, 'Date'].dt.month < cp8.loc[mask_charged, 'DOB'].dt.month) |
    (
        (cp8.loc[mask_charged, 'Date'].dt.month == cp8.loc[mask_charged, 'DOB'].dt.month) &
        (cp8.loc[mask_charged, 'Date'].dt.day   <  cp8.loc[mask_charged, 'DOB'].dt.day)
    )
)

In [8]:
print(before_birthday)

Series([], dtype: bool)


In [9]:
age_years = (
    cp8.loc[mask_charged, 'Date'].dt.year - cp8.loc[mask_charged, 'DOB'].dt.year
    - before_birthday.astype(int)
)

In [10]:
print(age_years)

Series([], dtype: int64)


In [11]:
# Write Age, NaN where not charged per your rule
cp8['Age'] = np.nan
cp8.loc[mask_charged, 'Age'] = age_years

In [12]:
# Quick peek
print(cp8[['DOB','Date','Age']].head(10))
print(cp8['Age'].describe())


  DOB                Date  Age
0 NaT 2026-06-01 23:53:00  NaN
1 NaT 2026-06-01 23:52:00  NaN
2 NaT 2026-06-01 23:29:00  NaN
3 NaT 2026-06-01 23:16:00  NaN
4 NaT 2026-06-01 23:15:00  NaN
5 NaT 2026-06-01 22:28:00  NaN
6 NaT 2026-06-01 22:16:00  NaN
7 NaT 2026-06-01 22:15:00  NaN
8 NaT 2026-06-01 22:03:00  NaN
9 NaT 2026-06-01 21:49:00  NaN
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: Age, dtype: float64


In [13]:
valid_age_df = cp8.loc[cp8['Age'].notna(), ['DOB', 'Date', 'Age']]

print(valid_age_df.head(20))   # just a preview

Empty DataFrame
Columns: [DOB, Date, Age]
Index: []


In [14]:
save_path = os.path.join(
    notebook_dir,
    "data",
    "missing_dates_csv",
    "md_checkpoints",
    "checkpoint9_age.csv"
)

cp8.to_csv(save_path, index=False)
print(f"Saved with Age column to: {save_path}")

Saved with Age column to: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/data/missing_dates_csv/md_checkpoints/checkpoint9_age.csv
